# Pathwiser 01: dataset analysis and verification of the model-ready data

**Input:** `data/processed/v1/{train,val,test}.jsonl` (output of `pathwiser benchmark build`, the validated benchmark)
**Verified output:** `data/model/v1/{train,val,test}.jsonl` (output of `pathwiser benchmark records`, listwise, model-ready)

This notebook **does not train a model** and does not generate data. It inspects the data, measures shortcut and leakage risks and
independently verifies the preprocessing output.

**Outline**
1. Setup and integrity check (manifest SHA-256)
2. Loading
3. Exploratory analysis (EDA)
4. Leakage and structural checks
5. Shortcut probes (*train + val only*)
6. Preprocessing and model-ready data (reproducibility, invariants, weights, lengths)
7. Helpers that feed a model (listwise collate / pairwise explode), demonstration only

**Rule:** the test set never enters a decision. For test there are only descriptive counts and structural checks.

## 1. Setup and integrity

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import random
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)


def find_root(start: Path | None = None) -> Path:
    start = start or Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "pathwiser").is_dir():
            return p
    raise FileNotFoundError("repo root not found (expected pyproject.toml and pathwiser/)")


ROOT = find_root()
sys.path.insert(0, str(ROOT))      # import the package from the checkout (no installation needed)

CFG = {
    "src_dir": "data/processed/v1",
    "model_dir": "data/model/v1",
    "seed": 42,
    "max_k": 10,
}
SPLITS = ["train", "val", "test"]
SRC = ROOT / CFG["src_dir"]
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])

# Plot style: fixed categorical order (blue, orange, aqua, ...), thin lines, faint grid.
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6", "#008300", "#e34948"]
SPLIT_COLOR = {"train": PAL[0], "val": PAL[1], "test": PAL[2]}
INK, INK2 = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.axisbelow": True, "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "legend.frameon": False,
})
print("root:", ROOT)
print("source:", SRC, "| exists:", SRC.exists())

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


src_manifest = json.loads((SRC / "manifest.json").read_text(encoding="utf-8"))
for name in ["train.jsonl", "val.jsonl", "test.jsonl", "conditions.jsonl"]:
    got, want = sha256_file(SRC / name), src_manifest["files"][name]["sha256"]
    assert got == want, f"{name}: checksum mismatch (dataset changed since the build)"
    print(f"OK  {name:18s} rows={src_manifest['files'][name]['rows']:>8,}  sha256={got[:16]}…")
print("dataset version:", src_manifest["version"], "| seed:", src_manifest["seed"], "| config_hash:", src_manifest["config_hash"][:16], "…")

## 2. Loading

Loading 200 MB+ of JSONL as is would be wasteful: from every record only the fields needed for preprocessing and analysis are kept.
`rec` = the slimmed-down raw fields (including lists), `df` = scalar columns only (for analysis).

In [ ]:
def feat_slice(slices: list[str], prefix: str, default: str = "") -> str:
    for s in slices:
        if s.startswith(prefix + ":"):
            return s.split(":", 1)[1]
    return default


def load_split(split: str) -> list[dict]:
    out = []
    with open(SRC / f"{split}.jsonl", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            cids = [c["cid"] for c in r["candidates"]]
            meta = r["candidate_meta"]
            assert [m["cid"] for m in meta] == cids, "candidate_meta order differs from candidates"
            c = r["construction"]
            out.append({
                "example_id": r["example_id"], "file_split": split, "split": r["split"], "tier": r["tier"],
                "dataset": r["source"]["dataset"], "dialogue_id": r["source"]["dialogue_id"], "group": r["split_group_id"],
                "context": r["context"], "message": r["current_message"], "node": r["node"],
                "cids": cids, "conditions": [x["description"] for x in r["candidates"]],
                "gold_cid": r["gold_cid"], "label_type": r["label_type"],
                "gold_idx": cids.index(r["gold_cid"]) if r["gold_cid"] is not None else -1,
                "uids": [m["condition_uid"] for m in meta], "groups": [m["equivalence_group_id"] for m in meta], "families": [m["family"] for m in meta],
                "roles": [m["role"] for m in meta], "strategies": [m["negative_strategy"] for m in meta],
                "slices": r["slices"], "kind": c["kind"], "label_risk": c["label_risk"], "n_cand": c["n_candidates"],
                "gold_position": c["gold_position"], "desc_variant": c["description_variant"],
                "paired_with": c["paired_with"], "removed_gold_uid": c["removed_gold_uid"],
                "n_context_total": r["features"]["n_context_total"],
            })
    return out


REC = {s: load_split(s) for s in SPLITS}
print({s: len(v) for s, v in REC.items()})


def scalar_row(r: dict) -> dict:
    sl = r["slices"]
    return {
        "example_id": r["example_id"], "split": r["split"], "file_split": r["file_split"], "tier": r["tier"],
        "dataset": r["dataset"], "label_type": r["label_type"],
        "nm_kind": r["kind"] if r["label_type"] == "no_match" else "",
        "label_risk": r["label_risk"], "k": r["n_cand"], "gold_idx": r["gold_idx"],
        "is_paired": r["paired_with"] is not None, "paired_with": r["paired_with"], "desc_variant": r["desc_variant"],
        "cond_seen": feat_slice(sl, "cond"), "svc_seen": feat_slice(sl, "svc"), "ctx_kind": feat_slice(sl, "ctx"),
        "ctx_turns": len(r["context"]), "ctx_words": sum(len(t["content"].split()) for t in r["context"]),
        "msg_words": len(r["message"].split()),
        "gold_words": len(r["conditions"][r["gold_idx"]].split()) if r["gold_idx"] >= 0 else np.nan,
        "max_cond_words": max(len(c.split()) for c in r["conditions"]),
        "node_origin": r["node"]["origin"], "node_name": r["node"]["name"], "dialogue_id": r["dialogue_id"],
        "message": r["message"],
    }


df = pd.DataFrame([scalar_row(r) for s in SPLITS for r in REC[s]])
df["split"] = pd.Categorical(df["split"], SPLITS, ordered=True)
DATASETS = sorted(df["dataset"].unique())
DS_COLOR = {d: PAL[i] for i, d in enumerate(DATASETS)}
print("datasets:", DATASETS)
df.head(3)

## 3. Exploratory analysis (EDA)

> **Reading rule:** a "main" example is one with an empty `paired_with`. SGD-X paraphrase twins (`paired_with` set) exist only in val/test
> and measure **the same moment a second time**; report metrics for main examples and twins **separately**, otherwise SGD is counted twice.

In [ ]:
main = df[~df.is_paired]
paired = df[df.is_paired]

print("== Example counts (main) ==")
tab = main.pivot_table(index="dataset", columns="split", values="example_id", aggfunc="count", fill_value=0, observed=False)[SPLITS]
tab["total"] = tab.sum(axis=1)
tab.loc["TOTAL"] = tab.sum()
display(tab)

print("== SGD-X paraphrase twins (val/test only) ==")
display(paired.pivot_table(index="dataset", columns="split", values="example_id", aggfunc="count", fill_value=0, observed=False)[["val", "test"]])

fig, ax = plt.subplots(figsize=(8, 3.4))
x = np.arange(len(DATASETS)); w = 0.26
for i, s in enumerate(SPLITS):
    vals = [int(tab.loc[d, s]) if d in tab.index else 0 for d in DATASETS]
    bars = ax.bar(x + (i - 1) * w, vals, w * 0.92, color=SPLIT_COLOR[s], label=s)
    ax.bar_label(bars, fmt="{:,.0f}", fontsize=7, color=INK2, padding=1.5, rotation=90)
ax.set_xticks(x, DATASETS); ax.set_ylabel("main examples"); ax.set_title("Dataset × split (main examples)")
ax.set_ylim(0, ax.get_ylim()[1] * 1.15); ax.legend(ncol=3, loc="upper left"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
print("== label_type share (main) ==")
nm = main.assign(no_match=(main.label_type == "no_match")).pivot_table(
    index="dataset", columns="split", values="no_match", aggfunc="mean", observed=False)[SPLITS].round(3)
display(nm)

print("== no_match kinds (main, all splits) ==")
display(main[main.label_type == "no_match"].pivot_table(index="dataset", columns="nm_kind", values="example_id", aggfunc="count", fill_value=0))

print("== label_risk (main) ==")
display(main.pivot_table(index="label_risk", columns="split", values="example_id", aggfunc="count", fill_value=0, observed=False)[SPLITS])

In [ ]:
# candidate-set size (k) distribution and gold position
kdist = main.pivot_table(index="k", columns="split", values="example_id", aggfunc="count", fill_value=0, observed=False)[SPLITS]
kshare = kdist / kdist.sum()

fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
ks = list(kshare.index); x = np.arange(len(ks)); w = 0.26
for i, s in enumerate(SPLITS):
    axs[0].bar(x + (i - 1) * w, kshare[s].values, w * 0.92, color=SPLIT_COLOR[s], label=s)
axs[0].set_xticks(x, ks); axs[0].set_xlabel("number of candidates k"); axs[0].set_ylabel("share"); axs[0].set_title("Candidate-set size"); axs[0].legend(ncol=3, loc="upper right", fontsize=8)
axs[0].grid(axis="x", visible=False); axs[0].set_ylim(0, 0.26)

route = main[main.label_type == "route"]
pos_rows = []
for k, g in route.groupby("k"):
    counts = g.gold_idx.value_counts(normalize=True).reindex(range(k), fill_value=0)
    pos_rows.append({"k": k, "n": len(g), "max_dev_from_uniform": float((counts - 1 / k).abs().max()),
                     "pos0_share": float(counts.iloc[0]), "expected": 1 / k})
pos = pd.DataFrame(pos_rows).set_index("k")
axs[1].bar(pos.index.astype(str), pos["max_dev_from_uniform"], color=PAL[0], width=0.6)
axs[1].set_xlabel("k"); axs[1].set_ylabel("max |pay − 1/k|"); axs[1].set_title("Gold position: deviation from uniform"); axs[1].grid(axis="x", visible=False)
plt.tight_layout(); plt.show()
display(pos.round(4))

In [ ]:
print("== Context kind (main): dataset × ctx_kind ==")
display(main.pivot_table(index="dataset", columns="ctx_kind", values="example_id", aggfunc="count", fill_value=0))

print("== Message length (words) per dataset ==")
q = main.groupby("dataset")["msg_words"].describe(percentiles=[.5, .9, .99])[["count", "mean", "50%", "90%", "99%", "max"]].round(1)
display(q)
print("== Context length (words; examples with context) ==")
display(main[main.ctx_turns > 0].groupby("dataset")["ctx_words"].describe(percentiles=[.5, .9, .99])[["count", "mean", "50%", "90%", "99%", "max"]].round(1))
print("== Candidate description length (longest candidate, words) ==")
display(main.groupby("dataset")["max_cond_words"].describe(percentiles=[.5, .9, .99])[["mean", "50%", "90%", "99%", "max"]].round(1))

In [ ]:
print("== Generalisation slices (main, route): cond:seen/unseen and svc:seen/unseen ==")
rt = main[main.label_type == "route"]
display(rt.pivot_table(index=["split", "cond_seen"], columns="dataset", values="example_id", aggfunc="count", fill_value=0, observed=True))

print("== Description variant × split (all examples) ==")
display(df.pivot_table(index="desc_variant", columns="split", values="example_id", aggfunc="count", fill_value=0, observed=False)[SPLITS])

### EDA summary: what to watch for
(Read the output of the cells above together with these notes.)

* The datasets differ a lot in size: SGD dominates. The raw distribution would weight training towards SGD; §6 applies weighting.
* In the single-turn datasets (Banking77/CLINC/HWU) the context is empty; only SGD/STAR/ABCD have real multi-turn context. "Context use" can therefore only be measured on those.
* The `cond:unseen` / `svc:unseen` slices exist in val/test. The project's central question (generalisation) must be reported with these slices.

## 4. Leakage and structural checks

These are **hard** checks: if one fails, the notebook stops. Looking at the test set here is a structural check only; it does not enter preprocessing decisions.

In [ ]:
def check(cond: bool, msg: str) -> None:
    print(("OK   " if cond else "FAIL ") + msg)
    assert cond, msg


ALL = [r for s in SPLITS for r in REC[s]]

# --- identity and split consistency
check(len({r["example_id"] for r in ALL}) == len(ALL), "example_id is unique across all splits")
check(all(r["split"] == r["file_split"] for r in ALL), "split inside the record == file name")

# --- structure
bad = []
for r in ALL:
    k = len(r["cids"])
    ok = (k == r["n_cand"] and 2 <= k <= CFG["max_k"] and len(set(r["cids"])) == k
          and len(set(r["conditions"])) == k and all(c.strip() for c in r["conditions"]) and r["message"].strip())
    if r["label_type"] == "route":
        ok = ok and r["gold_cid"] in r["cids"] and r["gold_idx"] >= 0 and r["roles"].count("gold") == 1 and r["roles"][r["gold_idx"]] == "gold"
    else:
        ok = ok and r["gold_cid"] is None and "gold" not in r["roles"]
    if not ok:
        bad.append(r["example_id"])
check(not bad, f"structure: gold in candidates, no gold in no_match, descriptions unique and non-empty, 2 <= k <= {CFG['max_k']} (violations={len(bad)})")

# is gold_position 0- or 1-based?
route_all = [r for r in ALL if r["label_type"] == "route"]
offsets = Counter(r["gold_position"] - r["gold_idx"] for r in route_all)
print("gold_position - gold_idx offset:", dict(offsets))
check(len(offsets) == 1, "gold_position is self-consistent (constant offset)")

# --- no dialogue leakage in the multi-turn datasets
multi = ["sgd", "star", "abcd"]
dlg = {s: {r["dialogue_id"] for r in REC[s] if r["dataset"] in multi} for s in SPLITS}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    check(not (dlg[a] & dlg[b]), f"no dialogue leakage: {a} ∩ {b} = {len(dlg[a] & dlg[b])}")


def norm_key(s: str) -> str:
    return re.sub(r"[^a-z0-9 ]+", "", s.lower()).strip()


# --- no text leakage in the single-turn datasets (main examples)
# Contract (benchmark.splits.dedupe_single_turn): no repeats within a dataset and split; no copy of a text in a later split
# (in any single-turn dataset). The same text in DIFFERENT datasets within one split is kept on purpose (separate candidate pools).
single = ["banking77", "clinc150", "hwu64"]
msgs_ds = {s: [(r["dataset"], norm_key(r["message"])) for r in REC[s] if r["dataset"] in single and r["paired_with"] is None] for s in SPLITS}
check(all(len(v) == len(set(v)) for v in msgs_ds.values()), "single-turn messages are unique within (dataset, split)")
msgs = {s: {m for _, m in v} for s, v in msgs_ds.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    inter = msgs[a] & msgs[b]
    check(not inter, f"no single-turn text leakage (across all datasets): {a} ∩ {b} = {len(inter)}")
xds = {s: len(msgs_ds[s]) - len(msgs[s]) for s in SPLITS}
print("info: same text in different datasets within one split (kept on purpose):", xds)

In [ ]:
# --- paraphrase twins
by_id = {r["example_id"]: r for r in ALL}
pr = [r for r in ALL if r["paired_with"] is not None]
check(all(r["split"] in ("val", "test") for r in pr), "paired examples only in val/test")
check(all(r["paired_with"] in by_id for r in pr), "every paired example has its main twin")
ok = True
for r in pr:
    m = by_id[r["paired_with"]]
    ok &= (m["split"] == r["split"] and m["message"] == r["message"] and m["label_type"] == r["label_type"]
           and m["paired_with"] is None and m["desc_variant"] != r["desc_variant"]
           and (m["gold_idx"] >= 0) == (r["gold_idx"] >= 0))
    if m["gold_idx"] >= 0 and r["gold_idx"] >= 0:
        ok &= m["uids"][m["gold_idx"]] == r["uids"][r["gold_idx"]] or m["dataset"] == "sgd"   # SGD-X renames services/intents
check(ok, "paired example: same split, same message, same label_type, different description variant")

# --- description-variant policy: train only {orig,v1,v2,v3}; val/test twins use v4/v5
variants = df.groupby("split", observed=True)["desc_variant"].apply(lambda s: sorted(set(s)))
display(variants)
check(set(variants["train"]) <= {"orig", "v1", "v2", "v3"}, "train uses only the orig/v1/v2/v3 description variants")
check(set(df[(df.is_paired) & (df.split == "val")].desc_variant) <= {"v4"}, "val twins use only v4")
check(set(df[(df.is_paired) & (df.split == "test")].desc_variant) <= {"v5"}, "test twins use only v5")

# --- verify the 'cond:seen / cond:unseen' claim.
# Definition (benchmark.examples): seen   <=> the gold's EQUIVALENCE GROUP exists in the train pool (possibly another service or wording)
#                            unseen <=> its group is not in the train pool.
conds_src = [json.loads(l) for l in open(SRC / "conditions.jsonl", encoding="utf-8")]
train_pool_groups = {c["equivalence_group_id"] for c in conds_src if c["partition"] == "train"}
train_cond_uids = {u for r in REC["train"] for u in r["uids"]}                    # appear as candidate/gold in train
train_gold_uids = {r["uids"][r["gold_idx"]] for r in REC["train"] if r["gold_idx"] >= 0}

tally = Counter()
for s_ in ("val", "test"):
    for r in REC[s_]:
        if r["gold_idx"] < 0 or r["paired_with"] is not None:
            continue
        flag = feat_slice(r["slices"], "cond")
        g_uid, g_grp = r["uids"][r["gold_idx"]], r["groups"][r["gold_idx"]]
        tally[(flag, "group_in_train_pool" if g_grp in train_pool_groups else "group_not_in_train_pool")] += 1
        if flag == "seen":
            tally[("seen", "exact_same_condition_train_gold" if g_uid in train_gold_uids else "equivalent_condition_only")] += 1
        if flag == "unseen":
            tally[("unseen", "uid_in_train_candidates(VIOLATION)" if g_uid in train_cond_uids else "uid_not_in_train_candidates")] += 1
for k_, v_ in sorted(tally.items()):
    print(f"{k_[0]:7s} {k_[1]:34s} {v_:>7,}")
check(tally[("seen", "group_not_in_train_pool")] == 0, "cond:seen => gold group in train pool")
check(tally[("unseen", "group_in_train_pool")] == 0, "cond:unseen => gold group NOT in train pool")
check(tally[("unseen", "uid_in_train_candidates(VIOLATION)")] == 0, "cond:unseen => gold condition never appeared in train, not even as a candidate")
n_seen_all = sum(v_ for k_, v_ in tally.items() if k_[0] == "seen" and k_[1] in ("exact_same_condition_train_gold", "equivalent_condition_only"))
n_equiv = tally[("seen", "equivalent_condition_only")]
print(f"\nWARNING (reporting): {n_equiv / n_seen_all:.0%} of the 'cond:seen' examples only saw an EQUIVALENT condition (the same condition id is not a train gold).")
print("Reporting these as 'seen' makes generalisation look better than it is -> report them separately via meta.gold_seen_exact.")

In [ ]:
# --- does the node text give away the label?
print("== node_origin × label_type (train, main) ==")
display(main[main.split == "train"].pivot_table(index=["dataset", "node_origin"], columns="label_type", values="example_id", aggfunc="count", fill_value=0))
print("== Context present/absent × label_type (train, main): no_match share ==")
tr = main[main.split == "train"].assign(has_ctx=lambda d: d.ctx_turns > 0, nm=lambda d: d.label_type == "no_match")
display(tr.pivot_table(index="dataset", columns="has_ctx", values="nm", aggfunc="mean").round(3))

**An honest note on the node.** `node.name/description` is a *fixed* text per dataset in the single-turn datasets, so a model can learn the dataset identity
from this field. It is still part of Pathwiser's **product input contract**, so it is given to the model by default. Since the candidate descriptions already
reveal the dataset, removing the `dataset` column does *not* remove that information; it only closes one easy shortcut. The real test is the ablation with
`pathwiser benchmark records --no-node`. The table above shows whether the node gives away the **label** (route / no_match).

## 5. Shortcut probes (train + val only)

These are **not learned models**; they are counting and rule-based diagnostics. The question: can a model answer correctly without "understanding"?
The test set is deliberately **left out**. Chance = mean `1/k`.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_tok = re.compile(r"[a-z0-9']+")


def toks(s: str) -> set[str]:
    return {t for t in _tok.findall(s.lower()) if t not in ENGLISH_STOP_WORDS and len(t) > 1}


def tie_credit(scores: list[float], gold: int) -> float:
    best = max(scores)
    idx = [i for i, s in enumerate(scores) if s == best]
    return (1 / len(idx)) if gold in idx else 0.0


rng = random.Random(CFG["seed"])
PROBE_CAP = 15000          # cap per split × dataset (deterministic sampling)
probe_rows = []
train_stats = {"app": Counter(), "gold": Counter()}
for r in REC["train"]:
    for u, role in zip(r["uids"], r["roles"]):
        train_stats["app"][u] += 1
        train_stats["gold"][u] += role == "gold"

for s in ("train", "val"):
    pool = defaultdict(list)
    for r in REC[s]:
        if r["gold_idx"] >= 0 and r["paired_with"] is None:
            pool[r["dataset"]].append(r)
    for ds, items in pool.items():
        items = items if len(items) <= PROBE_CAP else rng.sample(items, PROBE_CAP)
        for r in items:
            m = toks(r["message"])
            ov = [len(m & toks(c)) for c in r["conditions"]]
            wc = [len(c.split()) for c in r["conditions"]]
            row = {"split": s, "dataset": ds, "k": r["n_cand"], "chance": 1 / r["n_cand"],
                   "overlap_top1": tie_credit(ov, r["gold_idx"]), "longest_top1": tie_credit(wc, r["gold_idx"]),
                   "prior_top1": np.nan}
            if s == "val" and all(train_stats["app"][u] > 0 for u in r["uids"]):
                prior = [train_stats["gold"][u] / train_stats["app"][u] for u in r["uids"]]
                row["prior_top1"] = tie_credit(prior, r["gold_idx"])
            probe_rows.append(row)

probe = pd.DataFrame(probe_rows)
summ = probe.groupby(["split", "dataset"])[["chance", "overlap_top1", "longest_top1", "prior_top1"]].mean().round(3)
summ["n"] = probe.groupby(["split", "dataset"]).size()
summ["prior_coverage"] = probe.groupby(["split", "dataset"])["prior_top1"].apply(lambda s: s.notna().mean()).round(2)
display(summ)

In [ ]:
v = summ.xs("val", level="split")
fig, ax = plt.subplots(figsize=(8.5, 3.6))
x = np.arange(len(v)); w = 0.2
series = [("chance", "chance (1/k)", "#9a9893"), ("overlap_top1", "word overlap (message-candidate)", PAL[0]),
          ("longest_top1", "longest candidate", PAL[1]), ("prior_top1", "condition prior rate (train)", PAL[2])]
for i, (col, lab, colr) in enumerate(series):
    vals = v[col].fillna(0).values
    bars = ax.bar(x + (i - 1.5) * w, vals, w * 0.92, color=colr, label=lab)
    ax.bar_label(bars, labels=[f"{a:.2f}" if a > 0 else "–" for a in vals], fontsize=7, color=INK2, padding=1.5)
ax.set_xticks(x, v.index); ax.set_ylim(0, 1.0); ax.set_ylabel("top-1 accuracy (route, main, val)")
ax.set_title("Accuracy of rules that need no model (how far above chance?)")
ax.legend(ncol=2, fontsize=8, loc="upper left"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**How to read this**

* If `overlap_top1` is far above chance, the dataset is largely solvable by word overlap, which weakens any claim that a model "understands". The TF-IDF baseline should therefore be read as a **sign of easiness**, not as a lower bound.
* `longest_top1` should be close to chance (the gold description is usually not the longest).
* `prior_top1` (val only, examples whose candidates were all seen in train; `prior_coverage` is their share): if far above chance, condition frequency acts as a prior. The probe does not apply to `unseen` conditions (that is why coverage < 1).
* The probes are informative; they do not make automatic decisions about the data.

## 6. Preprocessing and model-ready data

Preprocessing lives **in the package, not in the notebook**: `pathwiser/benchmark/records.py` (`pathwiser benchmark records`).
That makes it reproducible, testable and identical for every model. The notebook only **verifies** the output.

| Step | Decision | Why |
|---|---|---|
| Text normalisation | Unicode NFC, drop control and zero-width characters, collapse whitespace | Does not change the meaning; **no lower-casing or punctuation stripping** (unnecessary for transformers, TF-IDF lower-cases itself) |
| Context | Newest 6 turns; if the 160-word budget is exceeded the **oldest turns are dropped** | The current message and the latest turns are the most informative |
| Message | Never truncated to fit the context budget | The decision rests on this message |
| Serialisation | `query` = node + context + message, `conditions[i]` = description only | The condition **id appears nowhere** |
| Target | `label` in [0,k) or `-1` (no_match); `label_null` = k (virtual "none of them" slot) | Two forms for listwise softmax and threshold-based decisions |
| Weight | Train only: dataset weight ∝ n^(-0.5) × risky-label factor (0.5); mean 1 | Reduce the dominance of the large datasets |
| Evaluation | val/test weight 1.0; reporting by slice (`meta.slices`) | Slice-level generalisation results |
| Written | `input` (goes to the model), `target`, `weight`, `meta` (**never goes to the model**, analysis only) | Keep the leakage surface closed |

In [ ]:
# --- reproducibility: rebuild the records in a temporary directory and compare hashes with the stored output
import tempfile

from pathwiser.benchmark.records import ModelDataConfig, export
from pathwiser.storage.dataset import collate_listwise, explode_pairs, load_split

MODEL_DIR = ROOT / CFG["model_dir"]
model_manifest = json.loads((MODEL_DIR / "manifest.json").read_text(encoding="utf-8"))
with tempfile.TemporaryDirectory() as tmp:
    fresh = export(SRC, Path(tmp) / "v1", ModelDataConfig(**model_manifest["preprocessing"]), log=lambda s: None)
for name in ("train.jsonl", "val.jsonl", "test.jsonl"):
    check(fresh["files"][name]["sha256"] == model_manifest["files"][name]["sha256"], f"{name}: rebuild has the same hash as the stored output")
print("preprocessing options:", model_manifest["preprocessing"])
print("preprocessing statistics:", model_manifest["stats"])

In [ ]:
# --- load from disk and verify (trust the files, not in-memory objects)
LOADED = {s: load_split(s, MODEL_DIR) for s in SPLITS}
for s in SPLITS:
    check(len(LOADED[s]) == model_manifest["files"][f"{s}.jsonl"]["rows"], f"{s}: row count matches the manifest ({len(LOADED[s]):,})")
    check(sha256_file(MODEL_DIR / f"{s}.jsonl") == model_manifest["files"][f"{s}.jsonl"]["sha256"], f"{s}: SHA-256 matches the manifest")
check(model_manifest["source_files_sha256"] == {n: src_manifest["files"][n]["sha256"] for n in ("train.jsonl", "val.jsonl", "test.jsonl")},
      "model data was built from the verified source version")

TOP = {"example_id", "split", "input", "target", "weight", "meta"}
INPUT = {"node", "context", "message", "query", "conditions"}
src_by_id = {r["example_id"]: r for r in ALL}
idlike = re.compile(r"^[a-z0-9]+([_:][a-z0-9]+)+$")          # snake_case / uid-like identifiers

problems = Counter()
for s in SPLITS:
    for m in LOADED[s]:
        src = src_by_id[m["example_id"]]
        problems["top_keys"] += set(m) != TOP
        problems["input_keys"] += set(m["input"]) != INPUT
        k = len(m["input"]["conditions"])
        t = m["target"]
        problems["label_range"] += not (-1 <= t["label"] < k)
        problems["null_label"] += t["label_null"] != (t["label"] if t["label"] >= 0 else k)
        problems["has_match"] += t["has_match"] != int(t["label"] >= 0)
        problems["label_vs_source"] += (t["label"] != src["gold_idx"])
        problems["label_type_vs_target"] += (m["meta"]["label_type"] == "route") != (t["label"] >= 0)
        problems["split"] += m["split"] != s
        q = m["input"]["query"]
        problems["query_has_message"] += m["input"]["message"] not in q
        problems["query_has_no_cond_text"] += any(c in q for c in m["input"]["conditions"] if len(c) > 25)   # candidates must not leak into the query
        problems["uid_in_text"] += any(u in q or any(u in c for c in m["input"]["conditions"]) for u in src["uids"])
        problems["idlike_condition"] += any(idlike.match(c) for c in m["input"]["conditions"])
        problems["weight_nonpositive"] += not (m["weight"] > 0)
        problems["eval_weight_not_1"] += s != "train" and m["weight"] != 1.0
check(sum(problems.values()) == 0, f"model-ready record invariants (violations: {dict((k_, v_) for k_, v_ in problems.items() if v_)})")

tw = [m["weight"] for m in LOADED["train"] if not m["meta"]["is_paired"]]
check(abs(np.mean(tw) - 1.0) < 1e-3, f"mean train main weight ~ 1 ({np.mean(tw):.4f})")
check(not any(m["meta"]["is_paired"] for m in LOADED["train"]), "no paired examples in train")

In [ ]:
# --- weights: raw share vs. effective share (train, main examples)
tr = LOADED["train"]
wt = pd.DataFrame({"n_train_main": Counter(m["meta"]["dataset"] for m in tr)})
wt["raw_share"] = wt.n_train_main / wt.n_train_main.sum()
wt["dataset_weight"] = pd.Series(model_manifest["dataset_weights"])
eff = defaultdict(float)
for m in tr:
    eff[m["meta"]["dataset"]] += m["weight"]
wt["effective_share"] = pd.Series(eff) / sum(eff.values())
display(wt.sort_values("n_train_main", ascending=False).round(4))

ex = next(m for m in tr if m["meta"]["dataset"] == "sgd" and m["input"]["context"] and m["target"]["has_match"])
print("\n--- example record (SGD, with context) ---")
print(ex["input"]["query"])
print()
for i, c in enumerate(ex["input"]["conditions"]):
    print(f"  [{i}]{' <-- gold' if i == ex['target']['label'] else ''}  {c}")
print("\ntarget:", ex["target"], "| weight:", ex["weight"])

In [ ]:
# --- length budget (words; the subword token count is typically ~1.3x)
def qwords(m): return len(m["input"]["query"].split())
len_tab = pd.DataFrame({s: pd.Series([qwords(m) for m in LOADED[s]]).describe(percentiles=[.5, .9, .99]) for s in SPLITS}).T
len_tab["≈token p99"] = (len_tab["99%"] * 1.3).round(0)
print("query length (words)")
display(len_tab[["count", "mean", "50%", "90%", "99%", "max", "≈token p99"]].round(1))
cw = pd.Series([len(c.split()) for s in SPLITS for m in LOADED[s] for c in m["input"]["conditions"]])
print("condition description (words):", cw.describe(percentiles=[.5, .99]).round(1).to_dict())

pairs_n = {s: sum(len(m["input"]["conditions"]) for m in LOADED[s]) for s in SPLITS}
print("pairwise row count (example × candidate):", pairs_n)

## 7. Helpers that feed a model (demonstration, no training)

`pathwiser.storage.dataset.collate_listwise` turns a mini-batch into numpy arrays (framework independent); `explode_pairs` expands the same data into pairwise form
(for TF-IDF/LR, LightGBM and the cross-encoder's pair input).

In [ ]:
demo = LOADED["train"][:64]
lb = collate_listwise(demo, max_k=CFG["max_k"])
pw = explode_pairs(demo)
print("listwise:", {k_: (v_.shape if hasattr(v_, "shape") else (len(v_), len(v_[0]) if isinstance(v_[0], list) else "")) for k_, v_ in lb.items()})
print("pairwise:", {k_: (v_.shape if hasattr(v_, "shape") else len(v_)) for k_, v_ in pw.items()})
check(int(lb["cond_mask"].sum()) == len(pw["is_gold"]), "collate and pairwise contain the same number of candidates")
check(int(pw["is_gold"].sum()) == int((lb["label"] >= 0).sum()), "pairwise positives = number of route examples")
check(bool((lb["label_null"][lb["label"] < 0] == lb["cond_mask"].sum(1)[lb["label"] < 0]).all()), "label_null = k for no_match")

final = pd.DataFrame({
    "examples": {s: len(LOADED[s]) for s in SPLITS},
    "main": {s: sum(not m["meta"]["is_paired"] for m in LOADED[s]) for s in SPLITS},
    "paraphrase_twins": {s: sum(m["meta"]["is_paired"] for m in LOADED[s]) for s in SPLITS},
    "no_match_share(main)": {s: round(float(np.mean([m["target"]["has_match"] == 0 for m in LOADED[s] if not m["meta"]["is_paired"]])), 3) for s in SPLITS},
    "mean_k": {s: round(float(np.mean([len(m["input"]["conditions"]) for m in LOADED[s]])), 2) for s in SPLITS},
    "pairwise_rows": pairs_n,
})
display(final)
print("Model-ready data:", MODEL_DIR)
print("Next: notebooks/02_baselines.ipynb (baseline results).")